# CardGuard — Model Experiments

## Objective

Develop and compare multiple machine learning models
for credit-card fraud detection.

The primary objective is to detect fraudulent transactions
while controlling false positives.

## Evaluation strategy

- Training set: model fitting
- Validation set: model comparison and threshold tuning
- Test set: final unbiased evaluation only

## Primary metrics

- PR-AUC
- Recall
- Precision
- F1-score
- ROC-AUC
- Confusion Matrix

Accuracy is not used as the primary metric because
fraud detection is a highly imbalanced classification problem.

In [19]:
import os
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve
)

In [20]:
train_df = pd.read_csv(r'D:\Creditcard_fraud_detection\data\processed\train_features.csv')
val_df = pd.read_csv(r'D:\Creditcard_fraud_detection\data\processed\validation_features.csv')
test_df = pd.read_csv(r'D:\Creditcard_fraud_detection\data\processed\test_features.csv')

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (156011, 52)
Validation: (33431, 52)
Test: (33431, 52)


In [21]:
train_df['is_fraud'].value_counts()

is_fraud
0    150890
1      5121
Name: count, dtype: int64

In [22]:
TARGET = "is_fraud"

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_val = val_df.drop(columns=[TARGET])
y_val = val_df[TARGET]

X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

print("Training features:", X_train.shape)
print("Training target:", y_train.shape)

print("Validation features:", X_val.shape)
print("Validation target:", y_val.shape)

print("Test features:", X_test.shape)
print("Test target:", y_test.shape)

Training features: (156011, 51)
Training target: (156011,)
Validation features: (33431, 51)
Validation target: (33431,)
Test features: (33431, 51)
Test target: (33431,)


In [23]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

print("Legitimate transactions:", negative_count)
print("Fraudulent transactions:", positive_count)

print("Fraud percentage:",round((positive_count / len(y_train)) * 100, 4),"%")
scale_pos_weight = negative_count / positive_count
print("Scale Pos Weight:", scale_pos_weight)

Legitimate transactions: 150890
Fraudulent transactions: 5121
Fraud percentage: 3.2825 %
Scale Pos Weight: 29.464948252294473


In [24]:
X_train.columns

Index(['merchant', 'category', 'amt', 'first', 'last', 'gender', 'street',
       'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob',
       'unix_time', 'merch_lat', 'merch_long', 'transaction_hour',
       'transaction_day', 'transaction_dayofweek', 'transaction_month',
       'transaction_year', 'hour_sin', 'hour_cos', 'dayofweek_sin',
       'dayofweek_cos', 'is_weekend', 'is_night', 'is_business_hour', 'age',
       'amt_log', 'amount_bucket', 'card_transaction_number',
       'previous_transaction_amt', 'time_since_previous_transaction_minutes',
       'card_merchant_count', 'is_new_merchant', 'card_city_count',
       'is_new_city', 'card_category_count', 'is_new_category',
       'customer_merchant_distance_km', 'previous_merch_lat',
       'previous_merch_long', 'distance_from_previous_location_km',
       'location_velocity_kmph', 'previous_avg_amount', 'previous_std_amount',
       'amount_vs_historical_avg', 'amount_zscore'],
      dtype='str')

In [25]:
forbidden_columns = [
    "cc_num",
    "trans_num",
    "Unnamed: 0",
    "first",
    "last",
    "street",
    "trans_date_trans_time",
    "dob"
]

X_train = X_train.drop(columns=forbidden_columns,errors="ignore")
X_val = X_val.drop(columns=forbidden_columns,errors="ignore")
X_test = X_test.drop(columns=forbidden_columns,errors="ignore")
print("Remaining features:", X_train.shape[1])

Remaining features: 47


In [26]:
categorical_candidates = [
    "merchant",
    "category",
    "gender",
    "city",
    "state",
    "job",
    "zip",
    "amount_bucket"
]

categorical_features = [
    col for col in categorical_candidates
    if col in X_train.columns
]
numeric_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

print("Categorical features:")
print(categorical_features)

print("\nNumber of categorical features:",len(categorical_features))
print("\nNumber of numerical features:",len(numeric_features))

Categorical features:
['merchant', 'category', 'gender', 'city', 'state', 'job', 'zip', 'amount_bucket']

Number of categorical features: 8

Number of numerical features: 39


In [27]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown='ignore',sparse_output=True))
])

preprocessor = ColumnTransformer([
    ('numerical', numerical_pipeline,numeric_features),
    ("categorical",categorical_pipeline, categorical_features)
])


In [28]:
base_model = Pipeline([
    ("preprocess",preprocessor),
    ("base model", DummyClassifier(strategy='most_frequent'))
    
])

base_model.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('base model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](47,)","['merchant','category','amt',...,'previous_std_amount', 'amount_vs_historical_avg','amount_zscore']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,47
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By

In [29]:

y_pred = base_model.predict(X_val)

In [30]:
print("accuracy_score", accuracy_score(y_val,y_pred ))
print("============================================")
print("precision_score",precision_score(y_val, y_pred, zero_division=0))
print("===============================================")
print("recall_score", recall_score(y_val, y_pred, zero_division=0))
print("===============================================")
print("F1 Score ", f1_score(y_val, y_pred, zero_division=0))
print("===============================================")
print("classification_report   ")
print(classification_report(y_val,y_pred, zero_division=0))

accuracy_score 0.9625198169363764
precision_score 0.0
recall_score 0.0
F1 Score  0.0
classification_report   
              precision    recall  f1-score   support

           0       0.96      1.00      0.98     32178
           1       0.00      0.00      0.00      1253

    accuracy                           0.96     33431
   macro avg       0.48      0.50      0.49     33431
weighted avg       0.93      0.96      0.94     33431



In [31]:
y_prob = base_model.predict_proba(X_val)[:, 1]


print("ROC-AUC  :", roc_auc_score(y_val, y_prob))
print("PR-AUC   :", average_precision_score(y_val, y_prob))

ROC-AUC  : 0.5
PR-AUC   : 0.037480183063623586


<!-- The PR-AUC ≈ 0.006437 is essentially the fraud prevalence, which is what we'd expect from this no-skill majority-class baseline.

What this tells you

Your accuracy of 99.36% looks excellent, but the model catches:

0% of fraudulent transactions.

That's why accuracy is not an appropriate primary metric for CardGuard.

For the rest of your experiments, I recommend making PR-AUC, Recall, Precision and F1 your important metrics -->

In [ ]:
# What this tells you

# accuracy of 96% looks excellent but the model catches
# 0% of fraudulent transactions

# That is why accuracy is not an appropriate primary metric for CardGuard

# For the rest of your experiments, I recommend making PR-AUC, Recall, Precision and F1 your important metrics

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(class_weight='balanced',
                                             max_iter=1000,random_state=42),
    
    "RandomForestClassifier": RandomForestClassifier(n_estimators=300,class_weight="balanced",
                                                     n_jobs=-1,random_state=42),
    
    "XGBoost": XGBClassifier(n_estimators=300,max_depth = 6, learning_rate = 0.05,
                               subsample = 0.8, colsample=0.8, scale_pos_weight=scale_pos_weight,
                               eval_metric = "logloss", n_jobs = -1, random_state =42),
    
    "LightGBM": LGBMClassifier(n_estimators=300,learning_rate=0.05,max_depth=-1,num_leaves=31,
                               subsample=0.8,colsample_bytree=0.8,scale_pos_weight=scale_pos_weight,
                               n_jobs=-1,random_state=42,verbosity=-1)
}

In [ ]:
for model_name, model in models.items():
    print("="*50)
    print(f"Training: {model_name}")
    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    model_pipeline.fit(X_train,y_train)
    train_pred = model_pipeline.predict(X_train)
    val_pred  = model_pipeline.predict(X_val)
    
    print("==============Train Results================")
    print("accuracy score", accuracy_score(y_train,train_pred))
    print("precision_score", precision_score(y_train, train_pred,zero_division=0))
    print("recall score", recall_score(y_train, train_pred,zero_division=0))
    print("f1 score", f1_score(y_train, train_pred,zero_division=0))
    print("classification_report")
    print(classification_report(y_train, train_pred,zero_division=0))
    
    
    print("==============Validation Results================")
    print("accuracy score", accuracy_score(y_val, val_pred))
    print("precisiion reprot", precision_score(y_val,val_pred,zero_division=0))
    print("recall score", recall_score(y_val, val_pred,zero_division=0))
    print("f1 score",f1_score(y_val,val_pred,zero_division=0))
    print("classification report")
    print(classification_report(y_val,val_pred,zero_division=0))

Training: LogisticRegression
==============Train Results================
accuracy score 0.9701495407375121
precision_score 0.5241013920631623
recall score 0.9851591486037883
f1 score 0.6842069573472571
classification_report
              precision    recall  f1-score   support

           0       1.00      0.97      0.98    150890
           1       0.52      0.99      0.68      5121

    accuracy                           0.97    156011
   macro avg       0.76      0.98      0.83    156011
weighted avg       0.98      0.97      0.97    156011

==============Validation Results================
accuracy score 0.952140229128653
precisiion reprot 0.4238032498902064
recall score 0.7701516360734237
f1 score 0.546742209631728
classification report
              precision    recall  f1-score   support

           0       0.99      0.96      0.97     32178
           1       0.42      0.77      0.55      1253

    accuracy                           0.95     33431
   macro avg       0.71      0.

In [ ]:
comparison = []

for model_name, model in models.items():
    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    model_pipeline.fit(X_train, y_train)
    val_pred = model_pipeline.predict(X_val)
    val_prob = model_pipeline.predict_proba(X_val)[:, 1]
    comparison.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_val, val_pred),
        "Precision": precision_score(y_val, val_pred, zero_division=0),
        "Recall": recall_score(y_val, val_pred, zero_division=0),
        "F1": f1_score(y_val, val_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_val, val_prob),
        "PR-AUC": average_precision_score(y_val, val_prob)
    })

comparison_df = pd.DataFrame(comparison)

comparison_df

,Model,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,LogisticRegression,0.952140,0.423803,0.770152,0.546742,0.955423,0.645304
1,RandomForestClassifier,0.991206,0.982880,0.778931,0.869101,0.996733,0.967099
2,XGBoost,0.997398,0.961234,0.969673,0.965435,0.999592,0.993697
3,LightGBM,0.997278,0.958202,0.969673,0.963903,0.999619,0.994355


In [36]:
comparison_df.sort_values(
    by="PR-AUC",
    ascending=False
).reset_index(drop=True)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,LightGBM,0.997278,0.958202,0.969673,0.963903,0.999619,0.994355
1,XGBoost,0.997398,0.961234,0.969673,0.965435,0.999592,0.993697
2,RandomForestClassifier,0.991206,0.982880,0.778931,0.869101,0.996733,0.967099
3,LogisticRegression,0.952140,0.423803,0.770152,0.546742,0.955423,0.645304


In [38]:
model_xgb = Pipeline([
    ('preprocessor', preprocessor),
    ('model',XGBClassifier(n_estimators=300,max_depth = 6, learning_rate = 0.05,
                               subsample = 0.8, colsample=0.8, scale_pos_weight=scale_pos_weight,
                               eval_metric = "logloss", n_jobs = -1, random_state =42))
])


model_xgb.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](47,)","['merchant','category','amt',...,'previous_std_amount', 'amount_vs_historical_avg','amount_zscore']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,47
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By sp

In [41]:
y_pred_xgb = model_xgb.predict(X_test)

print("accuracy score", accuracy_score(y_test,y_pred_xgb))
print("precision_score", precision_score(y_test, y_pred_xgb,zero_division=0))
print("recall score", recall_score(y_test, y_pred_xgb,zero_division=0))
print("f1 score", f1_score(y_test, y_pred_xgb,zero_division=0))
print("classification_report")
print(classification_report(y_test, y_pred_xgb,zero_division=0))


accuracy score 0.9969788519637462
precision_score 0.947871416159861
recall score 0.9637809187279152
f1 score 0.9557599649583881
classification_report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     32299
           1       0.95      0.96      0.96      1132

    accuracy                           1.00     33431
   macro avg       0.97      0.98      0.98     33431
weighted avg       1.00      1.00      1.00     33431



In [ ]:
import joblib

In [45]:
joblib.dump(model_xgb,r"D:\Creditcard_fraud_detection\data\save model\cardguard_xgboost.pkl")
print("Model saved successfully!")

Model saved successfully!


In [48]:
model_xgb = joblib.load(r"D:\Creditcard_fraud_detection\data\save model\cardguard_xgboost.pkl")

print("Model loaded successfully!")

Model loaded successfully!
